In [1]:
import pandas as pd
import numpy as np
import requests

In [99]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=r"C:\Users\Shiva\Projects\climate-risk-intelligence\.env")
api_key = os.getenv("OPENAQ_API_KEY")

In [101]:
import requests

headers = {"X-API-Key": api_key}

response = requests.get("https://api.openaq.org/v3/locations?country_id=IN&limit=100", headers=headers)

print(response.status_code)
print(response.json())

200
{'meta': {'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 100, 'found': '>100'}, 'results': [{'id': 3, 'name': 'NMA - Nima', 'locality': None, 'timezone': 'Africa/Accra', 'country': {'id': 152, 'code': 'GH', 'name': 'Ghana'}, 'owner': {'id': 4, 'name': 'Unknown Governmental Organization'}, 'provider': {'id': 209, 'name': 'Dr. Raphael E. Arku and Colleagues'}, 'isMobile': False, 'isMonitor': True, 'instruments': [{'id': 2, 'name': 'Government Monitor'}], 'sensors': [{'id': 6, 'name': 'pm10 µg/m³', 'parameter': {'id': 1, 'name': 'pm10', 'units': 'µg/m³', 'displayName': 'PM10'}}, {'id': 5, 'name': 'pm25 µg/m³', 'parameter': {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': 'PM2.5'}}], 'coordinates': {'latitude': 5.58389, 'longitude': -0.19968}, 'licenses': None, 'bounds': [-0.19968, 5.58389, -0.19968, 5.58389], 'distance': None, 'datetimeFirst': None, 'datetimeLast': None}, {'id': 4, 'name': 'NMT - Nima', 'locality': None, 'timezone': 'Africa/Accra', 'country': {'id

In [102]:
import requests

headers = {"X-API-Key": api_key}

response = requests.get("https://api.openaq.org/v3/locations?&limit=100&parameters_id=2",headers=headers)
print(response.status_code)
print(response.json().keys())

200
dict_keys(['meta', 'results'])


In [ ]:
collection_list = []
for x in range(1,11):
    response = requests.get(f"https://api.openaq.org/v3/locations?country_id=IN&limit=100&parameters_id=2&page={x}",headers=headers)
    collection = response.json()["results"]
    collection_list.extend(collection)
df = pd.DataFrame(collection_list)
india_df = df[df["country"].apply(lambda x: x["code"]) == "IN"]
print(f"Indian locations: {len(india_df)}")


Indian locations: 25


In [ ]:
print(df.columns)

Index(['location_id', 'pm25_avg', 'pm10_avg'], dtype='object')


In [ ]:
print(india_df["name"].values)

['SPARTAN - IIT Kanpur' 'Delhi Technological University, Delhi - CPCB'
 'IGI Airport' 'Civil Lines' 'R K Puram, Delhi - DPCC'
 'Punjabi Bagh, Delhi - DPCC' 'Income Tax Office, Delhi - CPCB'
 'Anand Vihar, New Delhi - DPCC' 'Mandir Marg, Delhi - DPCC'
 'Collectorate - Gaya - BSPCB' 'Vikas Sadan, Gurugram - HSPCB'
 'MD University, Rohtak - HSPCB' 'Lalbagh, DN Park' 'Alandur Bus Depot'
 'Zoo Park, Hyderabad - TSPCB' 'Peenya, Bengaluru - KSPCB'
 'IHBAS, Delhi - CPCB' 'BTM Layout, Bengaluru - KSPCB'
 'Haldia, Haldia - WBPCB' 'Collectorate Jodhpur - RSPCB'
 'Rabindra Bharati University, Kolkata - WBSPCB' 'Chandrapur'
 'Sanjay Palace, Agra - UPPCB' 'Victoria Memorial - WBSPCB'
 'Collectorate - Muzaffarpur - BSPCB']


In [ ]:
india_df.to_csv("../data/india_aqi_locations.csv", index=False)

In [ ]:
clean_df = pd.DataFrame()
clean_df["id"] = india_df["id"]
clean_df["name"] = india_df["name"]
clean_df["latitude"] = india_df["coordinates"].apply(lambda x: x["latitude"])
clean_df["longitude"] = india_df["coordinates"].apply(lambda x: x["longitude"])
print(clean_df)

      id                                           name   latitude  longitude
9     12                           SPARTAN - IIT Kanpur  26.519000  80.233000
10    13   Delhi Technological University, Delhi - CPCB  28.744000  77.120000
12    15                                    IGI Airport  28.560000  77.094000
13    16                                    Civil Lines  28.678700  77.226200
14    17                        R K Puram, Delhi - DPCC  28.563262  77.186937
36    50                     Punjabi Bagh, Delhi - DPCC  28.674045  77.131023
64   103                Income Tax Office, Delhi - CPCB  28.623500  77.249400
116  235                  Anand Vihar, New Delhi - DPCC  28.646835  77.316032
117  236                      Mandir Marg, Delhi - DPCC  28.634100  77.200500
134  286                    Collectorate - Gaya - BSPCB  24.748969  84.943839
145  301                  Vikas Sadan, Gurugram - HSPCB  28.450124  77.026305
169  346                  MD University, Rohtak - HSPCB  28.8760

In [ ]:
clean_df.to_csv("../data/india_aqi_locations.csv", index=False)

In [ ]:
import requests

headers = {"X-API-Key": api_key}
location_id = 12
response = requests.get(f"https://api.openaq.org/v3/locations/{location_id}/sensors",headers=headers)

print(response.status_code)
print(response.json())

401
{'message': 'Unauthorized. A valid API key must be provided in the X-API-Key header.'}


In [ ]:
response = requests.get(f"https://api.openaq.org/v3/locations/17/sensors", headers=headers)
print(response.status_code)
print(response.json())

401
{'message': 'Unauthorized. A valid API key must be provided in the X-API-Key header.'}


In [ ]:
import time

collection_list = []

for location_id, name in zip(clean_df["id"], clean_df["name"]):
    try:
        response = requests.get(
            f"https://api.openaq.org/v3/locations/{location_id}/sensors",
            headers=headers,
            timeout=30
        )
        if response.status_code == 200:
            for sensor in response.json()["results"]:
                if sensor["parameter"]["name"] == "pm25":
                    if sensor["summary"] is not None:
                        avg = sensor["summary"]["avg"]
                        collection_list.append((location_id, name, avg))
    except Exception as e:
        print(f"Failed for {name}: {e}")
    time.sleep(0.5)

df = pd.DataFrame(collection_list, columns=["location_id", "name", "pm25_avg"])
print(df)

Empty DataFrame
Columns: [location_id, name, pm25_avg]
Index: []


In [ ]:
print(clean_df["name"].head())

9                             SPARTAN - IIT Kanpur
10    Delhi Technological University, Delhi - CPCB
12                                     IGI Airport
13                                     Civil Lines
14                         R K Puram, Delhi - DPCC
Name: name, dtype: object


In [ ]:
pd.read_csv(r"C:\Users\Shiva\Projects\climate-risk-intelligence\data\india_aqi_locations.csv")

,id,name,latitude,longitude
0,12,SPARTAN - IIT Kanpur,26.519000,80.233000
1,13,"Delhi Technological University, Delhi - CPCB",28.744000,77.120000
2,15,IGI Airport,28.560000,77.094000
3,16,Civil Lines,28.678700,77.226200
4,17,"R K Puram, Delhi - DPCC",28.563262,77.186937
5,50,"Punjabi Bagh, Delhi - DPCC",28.674045,77.131023
6,103,"Income Tax Office, Delhi - CPCB",28.623500,77.249400
7,235,"Anand Vihar, New Delhi - DPCC",28.646835,77.316032
8,236,"Mandir Marg, Delhi - DPCC",28.634100,77.200500
9,286,Collectorate - Gaya - BSPCB,24.748969,84.943839


In [ ]:
print(clean_df.head())
print(clean_df.columns.tolist())

    id                                          name   latitude  longitude
9   12                          SPARTAN - IIT Kanpur  26.519000  80.233000
10  13  Delhi Technological University, Delhi - CPCB  28.744000  77.120000
12  15                                   IGI Airport  28.560000  77.094000
13  16                                   Civil Lines  28.678700  77.226200
14  17                       R K Puram, Delhi - DPCC  28.563262  77.186937
['id', 'name', 'latitude', 'longitude']


In [ ]:
clean_df = pd.DataFrame()
clean_df["id"] = india_df["id"]
clean_df["name"] = india_df["name"]
clean_df["latitude"] = india_df["coordinates"].apply(lambda x: x["latitude"])
clean_df["longitude"] = india_df["coordinates"].apply(lambda x: x["longitude"])
print(clean_df)

      id                                           name   latitude  longitude
9     12                           SPARTAN - IIT Kanpur  26.519000  80.233000
10    13   Delhi Technological University, Delhi - CPCB  28.744000  77.120000
12    15                                    IGI Airport  28.560000  77.094000
13    16                                    Civil Lines  28.678700  77.226200
14    17                        R K Puram, Delhi - DPCC  28.563262  77.186937
36    50                     Punjabi Bagh, Delhi - DPCC  28.674045  77.131023
64   103                Income Tax Office, Delhi - CPCB  28.623500  77.249400
116  235                  Anand Vihar, New Delhi - DPCC  28.646835  77.316032
117  236                      Mandir Marg, Delhi - DPCC  28.634100  77.200500
134  286                    Collectorate - Gaya - BSPCB  24.748969  84.943839
145  301                  Vikas Sadan, Gurugram - HSPCB  28.450124  77.026305
169  346                  MD University, Rohtak - HSPCB  28.8760

In [ ]:
import time

collection_list = []

for location_id, name in zip(clean_df["id"], clean_df["name"]):
    try:
        response = requests.get(
            f"https://api.openaq.org/v3/locations/{location_id}/sensors",
            headers=headers,
            timeout=30
        )
        if response.status_code == 200:
            for sensor in response.json()["results"]:
                if sensor["parameter"]["name"] == "pm25":
                    if sensor["summary"] is not None:
                        avg = sensor["summary"]["avg"]
                        collection_list.append((location_id, name, avg))
    except Exception as e:
        print(f"Failed for {name}: {e}")
    time.sleep(0.5)

df = pd.DataFrame(collection_list, columns=["location_id", "name", "pm25_avg"])
print(df)

Empty DataFrame
Columns: [location_id, name, pm25_avg]
Index: []


In [ ]:
pm25_avg_over_900 = df[(df["pm25_avg"] < 900) & (df["pm25_avg"] > 0)]
clean_df["location_id"] = clean_df["id"]
pm25_avg_filtered = np.round(pm25_avg_over_900.groupby(["location_id"])["pm25_avg"].mean().reset_index(),2)
merged_df = pd.merge(clean_df,pm25_avg_filtered,on="location_id",how="left")
print(merged_df)

     id                                           name   latitude  longitude  \
0    12                           SPARTAN - IIT Kanpur  26.519000  80.233000   
1    13   Delhi Technological University, Delhi - CPCB  28.744000  77.120000   
2    15                                    IGI Airport  28.560000  77.094000   
3    16                                    Civil Lines  28.678700  77.226200   
4    17                        R K Puram, Delhi - DPCC  28.563262  77.186937   
5    50                     Punjabi Bagh, Delhi - DPCC  28.674045  77.131023   
6   103                Income Tax Office, Delhi - CPCB  28.623500  77.249400   
7   235                  Anand Vihar, New Delhi - DPCC  28.646835  77.316032   
8   236                      Mandir Marg, Delhi - DPCC  28.634100  77.200500   
9   286                    Collectorate - Gaya - BSPCB  24.748969  84.943839   
10  301                  Vikas Sadan, Gurugram - HSPCB  28.450124  77.026305   
11  346                  MD University, 

In [ ]:
merged_df = merged_df.drop(columns="location_id")
merged_df = merged_df.dropna()
print(merged_df)

Empty DataFrame
Columns: [id, name, latitude, longitude, pm25_avg]
Index: []


In [ ]:
merged_df.to_csv(r"C:\Users\Shiva\Projects\climate-risk-intelligence\data\india_aqi_final.csv", index=False)

In [ ]:

features_df = pd.read_csv(r"C:\Users\Shiva\Projects\climate-risk-intelligence\data\india_aqi_final.csv")
print(features_df)

Empty DataFrame
Columns: [id, name, latitude, longitude, pm25_avg]
Index: []


In [ ]:
custom_bins = [0,30,60,90,120,250,float('inf')]
group_names = ['Good','Moderate','Unhealthy for sensitive groups','Unhealthy','Very Unhealthy','Hazardous']

features_df["risk_category"] = pd.cut(features_df["pm25_avg"],bins=custom_bins,labels = group_names,right=False)
features_df[["name", "pm25_avg", "risk_category"]]

,name,pm25_avg,risk_category


In [ ]:
features_df["pm25_normalized"] = np.round((features_df["pm25_avg"] - features_df["pm25_avg"].min()) / (features_df["pm25_avg"].max() - features_df["pm25_avg"].min()),2)
print(features_df.head())

Empty DataFrame
Columns: [id, name, latitude, longitude, pm25_avg, risk_category, pm25_normalized]
Index: []


In [ ]:
features_df["region"] = features_df.apply(lambda row: "East" if row['longitude'] > 80 else "West" if row['longitude'] < 76 else "North" if row['latitude'] > 26 and row['longitude'] > 76 and row['longitude'] < 80 else "South" if row['latitude'] < 22 and row['longitude'] > 76 and row['longitude'] < 80 else "Central",axis = 1)
print(features_df[["name","region","pm25_avg"]])

Empty DataFrame
Columns: [name, region, pm25_avg]
Index: []


In [ ]:

print(features_df)

Empty DataFrame
Columns: [id, name, latitude, longitude, pm25_avg, risk_category, pm25_normalized, region]
Index: []


In [ ]:
features_df.to_csv(r"C:\Users\Shiva\Projects\climate-risk-intelligence\data\india_features.csv", index=False)

In [ ]:
features_df["risk_score"] = np.round(features_df["pm25_normalized"] * 100,1)
print(features_df["risk_score"])

Series([], Name: risk_score, dtype: object)


In [ ]:
features_df.to_csv(r"C:\Users\Shiva\Projects\climate-risk-intelligence\data\india_features.csv", index=False)

In [ ]:
encoded_df = pd.get_dummies(features_df["region"],dtype=int)
features_df = pd.concat([features_df,encoded_df],axis=1)
print(features_df)

Empty DataFrame
Columns: [id, name, latitude, longitude, pm25_avg, risk_category, pm25_normalized, region, risk_score]
Index: []


In [ ]:
x = features_df[["latitude","longitude","East","West","North","South"]]
y = features_df[["risk_score"]]
print(x)
print(y)

KeyError: "['East', 'West', 'North', 'South'] not in index"

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)


In [ ]:
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

(16, 6) (5, 6) (16, 1) (5, 1)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100,random_state=42)
model.fit(x_train,y_train.values.ravel())
print(model)




RandomForestRegressor(random_state=42)


In [ ]:
y_pred = model.predict(x_test)
print(y_pred)
print(y_test)

[57.46 41.65 38.44 65.92 57.01]
    risk_score
0        100.0
17         4.0
15         9.0
1         62.0
8         43.0


In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test,y_pred)
print(mae)

25.512


In [ ]:
import time

collection_list_pm25 = []
collection_list_pm10 = []

for location_id, name in zip(clean_df["id"], clean_df["name"]):
    try:
        response = requests.get(
            f"https://api.openaq.org/v3/locations/{location_id}/sensors",
            headers=headers,
            timeout=30
        )
        if response.status_code == 200:
            for sensor in response.json()["results"]:
                if sensor["parameter"]["name"] == "pm25":
                    if sensor["summary"] is not None:
                        avg_pm25 = sensor["summary"]["avg"]
                        collection_list_pm25.append((location_id, name, avg_pm25))
                elif sensor["parameter"]["name"] == "pm10":
                    if sensor["summary"] is not None:
                        avg_pm10 = sensor["summary"]["avg"]
                        collection_list_pm10.append((location_id, name, avg_pm10))
    except Exception as e:
        print(f"Failed for {name}: {e}")
    time.sleep(0.5)

df1 = pd.DataFrame(collection_list_pm25, columns=["location_id", "name", "pm25_avg"])
df2 = pd.DataFrame(collection_list_pm10, columns=["location_id", "name", "pm10_avg"])



In [ ]:
pm25_avg_over_900 = df1[(df1["pm25_avg"] < 900) & (df1["pm25_avg"] > 0)]
pm25_avg_filtered = np.round(pm25_avg_over_900.groupby(["location_id"])["pm25_avg"].mean().reset_index(),2)


In [ ]:
pm10_avg_over_900 = df2[(df2["pm10_avg"] < 900) & (df2["pm10_avg"] > 0)]
pm10_avg_filtered = np.round(pm10_avg_over_900.groupby(["location_id"])["pm10_avg"].mean().reset_index(),2)

In [ ]:
df = pd.merge(pm25_avg_filtered,pm10_avg_filtered , on=["location_id"], how="left")
df = df.dropna()
print(df)

    location_id  pm25_avg  pm10_avg
1            17    117.44    217.33
2            50    123.52    218.71
3           103    141.59    221.33
4           235    129.00    318.64
5           236    113.03    209.49
7           301    114.57    113.32
8           346     89.97     77.44
11          407     54.74    109.53
15          697     41.51     81.88
16          703     90.80    200.21
17          716     34.60    143.99
18          854     64.74    135.76
19          860     52.38    128.36


In [ ]:
final_df = pd.merge(clean_df,df,on="location_id",how="left")
print(final_df)

     id                                           name   latitude  longitude  \
0    12                           SPARTAN - IIT Kanpur  26.519000  80.233000   
1    13   Delhi Technological University, Delhi - CPCB  28.744000  77.120000   
2    15                                    IGI Airport  28.560000  77.094000   
3    16                                    Civil Lines  28.678700  77.226200   
4    17                        R K Puram, Delhi - DPCC  28.563262  77.186937   
5    50                     Punjabi Bagh, Delhi - DPCC  28.674045  77.131023   
6   103                Income Tax Office, Delhi - CPCB  28.623500  77.249400   
7   235                  Anand Vihar, New Delhi - DPCC  28.646835  77.316032   
8   236                      Mandir Marg, Delhi - DPCC  28.634100  77.200500   
9   286                    Collectorate - Gaya - BSPCB  24.748969  84.943839   
10  301                  Vikas Sadan, Gurugram - HSPCB  28.450124  77.026305   
11  346                  MD University, 

In [ ]:
final_df = final_df.drop(columns="location_id")
final_df = final_df.dropna()
print(final_df)

     id                                           name   latitude  longitude  \
4    17                        R K Puram, Delhi - DPCC  28.563262  77.186937   
5    50                     Punjabi Bagh, Delhi - DPCC  28.674045  77.131023   
6   103                Income Tax Office, Delhi - CPCB  28.623500  77.249400   
7   235                  Anand Vihar, New Delhi - DPCC  28.646835  77.316032   
8   236                      Mandir Marg, Delhi - DPCC  28.634100  77.200500   
10  301                  Vikas Sadan, Gurugram - HSPCB  28.450124  77.026305   
11  346                  MD University, Rohtak - HSPCB  28.876028  76.619464   
14  407                    Zoo Park, Hyderabad - TSPCB  17.349694  78.451437   
18  697                         Haldia, Haldia - WBPCB  22.060470  88.109737   
19  703                   Collectorate Jodhpur - RSPCB  26.292064  73.037911   
20  716  Rabindra Bharati University, Kolkata - WBSPCB  22.627875  88.380400   
21  854                                 

In [ ]:
custom_bins = [0,30,60,90,120,250,float('inf')]
group_names = ['Good','Moderate','Unhealthy for sensitive groups','Unhealthy','Very Unhealthy','Hazardous']
final_df["risk_category_pm25"] = pd.cut(final_df["pm25_avg"],bins=custom_bins,labels = group_names,right=False)
final_df["risk_category_pm10"] = pd.cut(final_df["pm10_avg"],bins=custom_bins,labels = group_names,right=False)

final_df[["name", "pm25_avg", "pm10_avg","risk_category_pm25","risk_category_pm10"]]

,name,pm25_avg,pm10_avg,risk_category_pm25,risk_category_pm10
4,"R K Puram, Delhi - DPCC",117.44,217.33,Unhealthy,Very Unhealthy
5,"Punjabi Bagh, Delhi - DPCC",123.52,218.71,Very Unhealthy,Very Unhealthy
6,"Income Tax Office, Delhi - CPCB",141.59,221.33,Very Unhealthy,Very Unhealthy
7,"Anand Vihar, New Delhi - DPCC",129.00,318.64,Very Unhealthy,Hazardous
8,"Mandir Marg, Delhi - DPCC",113.03,209.49,Unhealthy,Very Unhealthy
10,"Vikas Sadan, Gurugram - HSPCB",114.57,113.32,Unhealthy,Unhealthy
11,"MD University, Rohtak - HSPCB",89.97,77.44,Unhealthy for sensitive groups,Unhealthy for sensitive groups
14,"Zoo Park, Hyderabad - TSPCB",54.74,109.53,Moderate,Unhealthy
18,"Haldia, Haldia - WBPCB",41.51,81.88,Moderate,Unhealthy for sensitive groups
19,Collectorate Jodhpur - RSPCB,90.80,200.21,Unhealthy,Very Unhealthy


In [ ]:
final_df["pm25_normalized"] = np.round((final_df["pm25_avg"] - final_df["pm25_avg"].min()) / (final_df["pm25_avg"].max() - final_df["pm25_avg"].min()),2)
final_df["pm10_normalized"] = np.round((final_df["pm10_avg"] - final_df["pm10_avg"].min()) / (final_df["pm10_avg"].max() - final_df["pm10_avg"].min()),2)
print(final_df)

     id                                           name   latitude  longitude  \
4    17                        R K Puram, Delhi - DPCC  28.563262  77.186937   
5    50                     Punjabi Bagh, Delhi - DPCC  28.674045  77.131023   
6   103                Income Tax Office, Delhi - CPCB  28.623500  77.249400   
7   235                  Anand Vihar, New Delhi - DPCC  28.646835  77.316032   
8   236                      Mandir Marg, Delhi - DPCC  28.634100  77.200500   
10  301                  Vikas Sadan, Gurugram - HSPCB  28.450124  77.026305   
11  346                  MD University, Rohtak - HSPCB  28.876028  76.619464   
14  407                    Zoo Park, Hyderabad - TSPCB  17.349694  78.451437   
18  697                         Haldia, Haldia - WBPCB  22.060470  88.109737   
19  703                   Collectorate Jodhpur - RSPCB  26.292064  73.037911   
20  716  Rabindra Bharati University, Kolkata - WBSPCB  22.627875  88.380400   
21  854                                 

In [ ]:
final_df["region"] = final_df.apply(lambda row: "East" if row['longitude'] > 80 else "West" if row['longitude'] < 76 else "North" if row['latitude'] > 26 and row['longitude'] > 76 and row['longitude'] < 80 else "South" if row['latitude'] < 22 and row['longitude'] > 76 and row['longitude'] < 80 else "Central",axis = 1)
print(final_df[["name","region","pm25_avg","pm10_avg"]])


                                             name region  pm25_avg  pm10_avg
4                         R K Puram, Delhi - DPCC  North    117.44    217.33
5                      Punjabi Bagh, Delhi - DPCC  North    123.52    218.71
6                 Income Tax Office, Delhi - CPCB  North    141.59    221.33
7                   Anand Vihar, New Delhi - DPCC  North    129.00    318.64
8                       Mandir Marg, Delhi - DPCC  North    113.03    209.49
10                  Vikas Sadan, Gurugram - HSPCB  North    114.57    113.32
11                  MD University, Rohtak - HSPCB  North     89.97     77.44
14                    Zoo Park, Hyderabad - TSPCB  South     54.74    109.53
18                         Haldia, Haldia - WBPCB   East     41.51     81.88
19                   Collectorate Jodhpur - RSPCB   West     90.80    200.21
20  Rabindra Bharati University, Kolkata - WBSPCB   East     34.60    143.99
21                                     Chandrapur  South     64.74    135.76

In [ ]:
final_df["risk_score_pm25"] = np.round(final_df["pm25_normalized"] * 100,1)
final_df["risk_score_pm10"] = np.round(final_df["pm10_normalized"] * 100,1)


In [ ]:
final_df["risk_score"] = ((final_df["risk_score_pm25"] + final_df["risk_score_pm10"])/2)
final_df = final_df.drop(columns="risk_score_pm25")
final_df = final_df.drop(columns="risk_score_pm10")

print(final_df)

     id                                           name   latitude  longitude  \
4    17                        R K Puram, Delhi - DPCC  28.563262  77.186937   
5    50                     Punjabi Bagh, Delhi - DPCC  28.674045  77.131023   
6   103                Income Tax Office, Delhi - CPCB  28.623500  77.249400   
7   235                  Anand Vihar, New Delhi - DPCC  28.646835  77.316032   
8   236                      Mandir Marg, Delhi - DPCC  28.634100  77.200500   
10  301                  Vikas Sadan, Gurugram - HSPCB  28.450124  77.026305   
11  346                  MD University, Rohtak - HSPCB  28.876028  76.619464   
14  407                    Zoo Park, Hyderabad - TSPCB  17.349694  78.451437   
18  697                         Haldia, Haldia - WBPCB  22.060470  88.109737   
19  703                   Collectorate Jodhpur - RSPCB  26.292064  73.037911   
20  716  Rabindra Bharati University, Kolkata - WBSPCB  22.627875  88.380400   
21  854                                 

In [ ]:
final_df.to_csv(r"C:\Users\Shiva\Projects\climate-risk-intelligence\data\india_features_v2.csv", index=False)

In [ ]:
encoded_df = pd.get_dummies(final_df["region"],dtype=int)
final_df = pd.concat([final_df,encoded_df],axis=1)
print(final_df)


     id                                           name   latitude  longitude  \
4    17                        R K Puram, Delhi - DPCC  28.563262  77.186937   
5    50                     Punjabi Bagh, Delhi - DPCC  28.674045  77.131023   
6   103                Income Tax Office, Delhi - CPCB  28.623500  77.249400   
7   235                  Anand Vihar, New Delhi - DPCC  28.646835  77.316032   
8   236                      Mandir Marg, Delhi - DPCC  28.634100  77.200500   
10  301                  Vikas Sadan, Gurugram - HSPCB  28.450124  77.026305   
11  346                  MD University, Rohtak - HSPCB  28.876028  76.619464   
14  407                    Zoo Park, Hyderabad - TSPCB  17.349694  78.451437   
18  697                         Haldia, Haldia - WBPCB  22.060470  88.109737   
19  703                   Collectorate Jodhpur - RSPCB  26.292064  73.037911   
20  716  Rabindra Bharati University, Kolkata - WBSPCB  22.627875  88.380400   
21  854                                 

In [ ]:
x = final_df[["latitude","longitude","pm25_normalized","pm10_normalized","East","West","North","South"]]
y = final_df[["risk_score"]]
print(x)
print(y)

     latitude  longitude  pm25_normalized  pm10_normalized  East  West  North  \
4   28.563262  77.186937             0.77             0.58     0     0      1   
5   28.674045  77.131023             0.83             0.59     0     0      1   
6   28.623500  77.249400             1.00             0.60     0     0      1   
7   28.646835  77.316032             0.88             1.00     0     0      1   
8   28.634100  77.200500             0.73             0.55     0     0      1   
10  28.450124  77.026305             0.75             0.15     0     0      1   
11  28.876028  76.619464             0.52             0.00     0     0      1   
14  17.349694  78.451437             0.19             0.13     0     0      0   
18  22.060470  88.109737             0.06             0.02     1     0      0   
19  26.292064  73.037911             0.53             0.51     0     1      0   
20  22.627875  88.380400             0.00             0.28     1     0      0   
21  19.950000  79.300000    

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)


(10, 8) (3, 8) (10, 1) (3, 1)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100,random_state=42)
model.fit(x_train,y_train.values.ravel())
print(model)


RandomForestRegressor(random_state=42)


In [ ]:
y_pred = model.predict(x_test)
print(y_pred)
print(y_test)

[17.75 46.37 67.99]
    risk_score
21        26.0
19        52.0
4         67.5


In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test,y_pred)
print(mae)

4.789999999999999


In [ ]:
df["country"].unique()

KeyError: 'country'